In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
np.set_printoptions(threshold=sys.maxsize)

In [ ]:
np.sqrt(8000)

In [ ]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=10,
    board_h=20,
    vanish_zone=4, # Extra rows above the visible board to capture piece spawns
)

CONFIG = Configuration(
    max_board_size_w=10,
    max_board_size_h=20,
)

# Train

In [ ]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [ ]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

In [ ]:
print(env.reset()[0]["boards"].shape)
print(env.reset()[0]["queues"].shape)
print(env.reset()[0]["queue_idx"].shape)
print(env.reset()[0]["placement_mask"].shape)

In [ ]:
boards = env.reset()[0]["boards"]

### Model

In [ ]:
env.observation_space["boards"].shape

In [ ]:
from src.models import TurboMinoEncoder

feature_extractor = TurboMinoEncoder(
    env.observation_space,
    T_CONFIG,
    CONFIG,
)

### Correct forward pass (current model)

The feature extractor returns a single tensor `(B, max_placements)` — one scalar per placement.
The env provides a `placement_mask` so the model masks out invalid/padded slots with `-1e9`.

In [ ]:
import torch

obs, _ = env.reset()
tensor_obs = {
    "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
    "queues": torch.as_tensor(obs["queues"], dtype=torch.float32).unsqueeze(0),
    "queue_idx": torch.as_tensor(obs["queue_idx"], dtype=torch.long).unsqueeze(0),
    "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
}
print("boards:", tensor_obs["boards"].shape)              # (1, M, H, W)
print("queues:", tensor_obs["queues"].shape)              # (1, 2, S, C)
print("queue_idx:", tensor_obs["queue_idx"].shape)        # (1, M)

values = feature_extractor(tensor_obs)
print("Output shape:", values.shape)                       # (1, M)
print("Mask (first 15):", tensor_obs["placement_mask"][0, :15])
print("Valid placements:", tensor_obs["placement_mask"].sum().item())
print("Placement values:\n", values)
print("Best placement:", values.argmax().item())

In [ ]:
# import time 
# import tqdm

# total_iters = 4*5

# print('warming up...')
# for i in tqdm.tqdm(range(total_iters*1000)):
#     obs, _ = env.reset()
#     tensor_obs = {
#         "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
#         "queues": torch.as_tensor(obs["queues"], dtype=torch.float32).unsqueeze(0),
#         "queue_idx": torch.as_tensor(obs["queue_idx"], dtype=torch.long).unsqueeze(0),
#         "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
#     }

#     values = feature_extractor(tensor_obs)


# t1 = time.time()
# for i in range(total_iters):
#     obs, _ = env.reset()
#     tensor_obs = {
#         "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
#         "queues": torch.as_tensor(obs["queues"], dtype=torch.float32).unsqueeze(0),
#         "queue_idx": torch.as_tensor(obs["queue_idx"], dtype=torch.long).unsqueeze(0),
#         "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
#     }

#     values = feature_extractor(tensor_obs)

# t2 = time.time()

# print(f"Average inference time over {total_iters} iterations: {(t2 - t1) / total_iters:.4f} seconds")

# PreTrain